In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Configuration de Selenium
chrome_options = Options()
chrome_options.add_argument("--headless")  # Mode sans interface graphique
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")

# Lancer le navigateur Selenium
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# Fonction pour récupérer les annonces d'une page
def get_page_data(url):
    try:
        print(f"🔄 Scraping {url}...")
        
        # Charger la page avec Selenium
        driver.get(url)
        time.sleep(3)  # Pause pour éviter un blocage

        # Récupérer le contenu HTML après le chargement des scripts
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        datas = []
        car_models = soup.find_all('div', class_="sc-1g3sn3w-3 fHaZgB")  # Modifier selon la classe exacte
        
        if not car_models:
            print("⚠️ Aucune annonce trouvée sur cette page.")
            return datas

        for model in car_models:
            spans = model.find_all('span', class_="sc-1x0vz2r-0 gSLYtF")
            if len(spans) < 9:
                continue  # Ignorer les annonces incomplètes

            Marque = spans[3].text.strip() if len(spans) > 3 else "None"
            Kilométrage = spans[2].text.strip()
            Model = spans[4].text.strip()
            Nombre_de_portes = spans[5].text.strip()
            Première_main = spans[7].text.strip()
            Puissance_fiscale = spans[8].text.strip()
            État = spans[9].text.strip()

            # Détection du type
            if "occasion" in spans[0].text.strip().lower():
                Type = "Occasion"
            elif "location" in spans[0].text.strip().lower():
                Type = "Location"
            else:
                Type = "None"

            # Deuxième série de spans
            spans_2 = model.find_all('span', class_="sc-1x0vz2r-0 kQHNss")
            transmission = spans_2[1].text.strip() if len(spans_2) > 1 else "None"
            fuel = spans_2[2].text.strip() if len(spans_2) > 2 else "None"

            datas.append({
                'Marque': Marque, 'Kilométrage': Kilométrage, 'Model': Model,
                'Nb_portes': Nombre_de_portes, 'Première_main': Première_main,
                'Puissance': Puissance_fiscale, 'État': État, 'Type': Type,
                'Transmission': transmission, 'Carburant': fuel
            })

        return datas

    except Exception as e:
        print(f"❌ Erreur lors du scraping de {url} : {e}")
        return []

# Liste des pages à scraper
base_url = "https://www.avito.ma/fr/maroc/voitures-%C3%A0_vendre?o="
nb_pages = 1100  # Limiter le scraping pour éviter les blocages

all_cars = []
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

for i in range(1041, nb_pages + 1):
    url = base_url + str(i)
    try:
        print(f"🔄✅ Scraping page {i}...")
        
        # Charger la page avec requests (timeout = 60s pour éviter l'erreur de délai)
        response = requests.get(url, headers=headers, timeout=60)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Récupérer les liens des annonces
        urls = soup.find_all('a', class_="sc-1jge648-0 jZXrfL") 

        for link in urls:
            car_url = link.get('href')
            if car_url:
                full_url = "https://www.avito.ma" + car_url if car_url.startswith("/") else car_url

                all_cars.extend(get_page_data(full_url))
                
                time.sleep(3)  # Pause pour éviter un blocage

    except requests.exceptions.Timeout:
        print(f"❌ Timeout lors du chargement de {url}")
    except requests.exceptions.RequestException as e:
        print(f"❌ Erreur réseau : {e}")

print(f"✅ Scraping terminé. {len(all_cars)} annonces récupérées.")



In [ ]:
print(all_cars[:500])


In [ ]:
import pandas as pd

# Convertir all_cars (qui est probablement une liste de dictionnaires) en DataFrame
df_all_cars = pd.DataFrame(all_cars)

# Ajouter les données de df_all_cars dans le fichier CSV existant
df_all_cars.to_csv("voitures_avito.csv", mode='a', header=not pd.io.common.file_exists("voitures_avito.csv"), index=False, encoding="utf-8")

print(f"✅ {len(all_cars)} annonces ajoutées au fichier 'voitures_avito.csv'.")
